# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset: {getattr(dataset.metadata, 'name', 'N/A')}\n{getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, fields, and columns with their @id
record_sets = list(dataset.metadata.record_sets)  # All record set objects
print("Available record sets and their fields:\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print("  Fields:")
    for field in rs.fields:
        fname = getattr(field, 'name', '<unnamed>')
        print(f"    - {fname} (@id: {field.id}, dataType: {getattr(field, 'data_type', '?')})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.id}")
    print("")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose all available record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet {record_set_id}, columns:", df.columns.tolist())
            print(df.head(), "\n")
    except Exception as e:
        print(f"Warning: Could not load records from {record_set_id}: {e}")

# Choose a record set for demonstration (for notebooks with only 1 will work automatically, otherwise set explicitly)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Using main RecordSet: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for analysis.


In [ ]:
# Set a numeric field and a group field based on the schema or data
# We'll try to select a likely numeric and group field automatically from field metadata

numeric_field_id = None
group_field_id = None

if main_record_set_id:
    rs = [rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id][0]
    # Try to select a numeric field
    for field in rs.fields:
        dtype = getattr(field, 'data_type', '').lower() if hasattr(field, 'data_type') else ''
        if dtype in ['integer', 'float', 'number']:
            if field.id in dataframes[main_record_set_id].columns:
                numeric_field_id = field.id
                break
    if not numeric_field_id:
        # Fallback: pick first field with likely numeric name
        for col in dataframes[main_record_set_id].columns:
            if any(x in col.lower() for x in ['score', 'weight', 'count', 'value', 'amount', 'intensity', 'mw', 'abundance', 'normalized']):
                numeric_field_id = col
                break

    # Try to select a group field (e.g., description, type, accession, or other text field)
    for field in rs.fields:
        dtype = getattr(field, 'data_type', '').lower() if hasattr(field, 'data_type') else ''
        if dtype in ['text', 'string']:
            if field.id in dataframes[main_record_set_id].columns:
                group_field_id = field.id
                break
    if not group_field_id:
        # Fallback: first column with typical grouping name
        for col in dataframes[main_record_set_id].columns:
            if any(x in col.lower() for x in ['type', 'group', 'desc', 'sample']):
                group_field_id = col
                break

else:
    print("No main record set found.")

# Inform user choices
print(f"Numeric field selected: {numeric_field_id}")
print(f"Group field selected: {group_field_id}")

if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]
    # Convert to numeric with coercion (in case of parsing issues)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No appropriate numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(data=dataframes[main_record_set_id], x=numeric_field_id, kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If we have a group field, plot groupwise mean
    if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10,4))
        order = dataframes[main_record_set_id].groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).index[:15]
        sns.barplot(
            data=dataframes[main_record_set_id], x=group_field_id, y=numeric_field_id,
            estimator='mean', ci=None, order=order
        )
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 15)")
        plt.xticks(rotation=75)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.